# Решение домашнего задания #24

[задача](../../tasks/hometask_24.ipynb)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from typing import List, Dict, Tuple
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

np.random.seed(42)
torch.manual_seed(42)

## Загрузка модели Phi3

Используем **Phi3 (Phi-3 mini 3.8B)** - компактную версию Phi-3, оптимизированную для эффективной работы с изображениями и текстом.

In [ ]:
# Загрузка модели LLaVA-NeXT (v1.6)
model_name = "microsoft/Phi-3-mini-128k-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print(f"Model uses: {model.dtype}")

## Загрузка датасета WNUT-17

Используем **WNUT-17** - датасет с разметкой для задачи распознавания именованных сущностей в неформальных текстах (например, в твитах). Мы будем просить модель **обосновывать** свои ответы, чтобы увидеть куда направлено внимание при рассуждении.

In [ ]:
dataset = load_dataset("leondz/wnut_17")

label_names = dataset['train'].features['ner_tags'].feature.names
print(f"\nLabels NER: {label_names}")

# Data analysis
print(f"\nDataset sizes:")
print(f"  Train: {len(dataset['train'])} examples")
print(f"  Validation: {len(dataset['validation'])} examples")
print(f"\nWNUT-17 features:")
print(f"  - Types: {', '.join([l for l in label_names if l != 'O'])}")

example = dataset['train'][0]

print(f"  Tokens: {example['tokens'][:10]}...")
print(f"  NER tags: {[label_names[tag] for tag in example['ner_tags'][:10]]}...")


## Подготовка данных для обучения

In [ ]:
class DEERStatistics:
    def __init__(self, dataset, label_names):
        self.label_names = label_names
        self.entity_types = self._extract_entity_types()
        
        self.token_entity_counts = defaultdict(lambda: defaultdict(int))
        self.token_total_counts = defaultdict(int)
        
        self._compute_statistics(dataset)
        
    def _extract_entity_types(self):
        types = set()
        for label in self.label_names:
            if label.startswith('B-') or label.startswith('I-'):
                types.add(label.split('-')[1])
        return sorted(list(types))
    
    def _compute_statistics(self, dataset):
        for example in dataset:
            tokens = example['tokens']
            tags = [self.label_names[tag] for tag in example['ner_tags']]
            
            for token, tag in zip(tokens, tags):
                token_lower = token.lower()
                self.token_total_counts[token_lower] += 1
                
                if tag != 'O':  # If this is an entity
                    entity_type = tag.split('-')[1]  # Remove B-/I- prefix
                    self.token_entity_counts[entity_type][token_lower] += 1
        
        print(f"Found {len(self.token_total_counts)} unique tokens")
        print(f"Entity types: {self.entity_types}")
        
    def get_token_score(self, token: str, entity_type: str) -> float:
        token_lower = token.lower()
        if token_lower not in self.token_total_counts:
            return 0.0
        
        entity_count = self.token_entity_counts[entity_type][token_lower]
        total_count = self.token_total_counts[token_lower]
        
        return entity_count / total_count if total_count > 0 else 0.0
    
    def get_most_informative_tokens(self, entity_type: str, top_k: int = 10):
        scores = []
        for token in self.token_entity_counts[entity_type].keys():
            score = self.get_token_score(token, entity_type)
            scores.append((token, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

deer_stats = DEERStatistics(dataset['train'], label_names)

print("Most informative tokens for each entity type:")
for entity_type in deer_stats.entity_types:
    top_tokens = deer_stats.get_most_informative_tokens(entity_type, top_k=5)
    print(f"\n{entity_type}:")
    for token, score in top_tokens:
        print(f"  {token}: {score:.3f}")

In [ ]:
class LabelGuidedRetrieval:
    def __init__(self, statistics: DEERStatistics):
        self.stats = statistics
        
    def compute_overlap_score(self, test_tokens: List[str], train_example: Dict) -> float:
        train_tokens = train_example['tokens']
        train_tags = [self.stats.label_names[tag] for tag in train_example['ner_tags']]
        
        # Determining entity types present in the train example
        entity_types_in_train = set()
        for tag in train_tags:
            if tag != 'O':
                entity_types_in_train.add(tag.split('-')[1])
        
        if not entity_types_in_train:
            return 0.0
        
        # Compute score based on overlap of informative tokens
        total_score = 0.0
        for token in test_tokens:
            for entity_type in entity_types_in_train:
                token_score = self.stats.get_token_score(token, entity_type)
                total_score += token_score
        
        # Normalize by the number of tokens
        return total_score / len(test_tokens) if test_tokens else 0.0
    
    def retrieve_examples(self, test_tokens: List[str], train_dataset, k: int = 3) -> List[Dict]:
        scores = []
        for idx, train_example in enumerate(train_dataset):
            score = self.compute_overlap_score(test_tokens, train_example)
            scores.append((score, idx, train_example))
        
        # Sort by score and take top-k
        scores.sort(key=lambda x: x[0], reverse=True)
        
        return [example for _, _, example in scores[:k]]

retriever = LabelGuidedRetrieval(deer_stats)

test_example = dataset['test'][0]
retrieved = retriever.retrieve_examples(test_example['tokens'], dataset['train'][:1000], k=3)

print("Test tokens:", test_example['tokens'][:10])
print(f"\nFound {len(retrieved)} relevant examples")
print("\nExample of a retrieved example:")
print("Tokens:", retrieved[0]['tokens'][:10])
print("Tags:", [label_names[tag] for tag in retrieved[0]['ner_tags'][:10]])

In [ ]:
def create_ner_prompt(test_tokens: List[str], examples: List[Dict], label_names: List[str]) -> str:
    prompt = """Task: Named Entity Recognition (NER) for WNUT-17 (emerging entities in noisy text)
Label each token with one of: O (outside), B-corporation, I-corporation, B-creative-work, I-creative-work, B-group, I-group, B-location, I-location, B-person, I-person, B-product, I-product

Note: This is noisy user-generated text with emerging entities.
Format: token1|label1 token2|label2 ...\n\n"""
    
    # Add few-shot examples
    prompt += "Examples:\n\n"
    for i, example in enumerate(examples, 1):
        tokens = example['tokens']
        tags = [label_names[tag] for tag in example['ner_tags']]
        
        # Form the token|label string
        labeled_tokens = [f"{tok}|{tag}" for tok, tag in zip(tokens, tags)]
        prompt += f"Example {i}:\n"
        prompt += " ".join(labeled_tokens) + "\n\n"
    
    # Add test query
    prompt += "Now label the following text:\n"
    prompt += " ".join(test_tokens) + "\n\nAnswer (format: token1|label1 token2|label2 ...):\n"
    
    return prompt


test_tokens = dataset['test'][0]['tokens'][:15]  # Take the first 15 tokens
retrieved_examples = retriever.retrieve_examples(test_tokens, dataset['train'][:1000], k=2)

prompt = create_ner_prompt(test_tokens, retrieved_examples, label_names)
print("Example prompt (first 800 characters):")
print(prompt[:800] + "...")

## Инференс с объяснениями

In [ ]:
def deer_ner_inference(test_example, retriever, model, tokenizer, train_dataset, k=3):
    test_tokens = test_example['tokens']
    
    retrieved_examples = retriever.retrieve_examples(test_tokens, train_dataset, k=k)
    
    prompt = create_ner_prompt(test_tokens, retrieved_examples, label_names)
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=False)
    output_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    
    # Parse NER predictions from model output - try multiple strategies
    predictions = []
    
    # Clean output text
    output_text = output_text.strip()
    
    lines = output_text.split('\n')
    parsed_pairs = []
    
    for line in lines:
        line = line.strip()
        if '|' in line:
            tokens_in_line = line.split()
            for item in tokens_in_line:
                if '|' in item:
                    parsed_pairs.append(item)
    
    if not parsed_pairs:
        all_tokens = output_text.replace('\n', ' ').split()
        for item in all_tokens:
            if '|' in item:
                parsed_pairs.append(item)
    
    # Extract predictions from parsed pairs
    for i, token in enumerate(test_tokens):
        if i < len(parsed_pairs):
            pair = parsed_pairs[i]
            if '|' in pair:
                # Extract the tag (everything after last |)
                predicted_tag = pair.split('|')[-1].strip()
                # Validate tag format
                valid_tags = ['O'] + [f"{prefix}-{entity}" 
                                     for prefix in ['B', 'I'] 
                                     for entity in deer_stats.entity_types]
                if predicted_tag in valid_tags:
                    predictions.append(predicted_tag)
                else:
                    # Try to fix common formatting issues
                    predicted_tag = predicted_tag.replace('_', '-')
                    if predicted_tag in valid_tags:
                        predictions.append(predicted_tag)
                    else:
                        predictions.append("O")
            else:
                predictions.append("O")
        else:
            # Model didn't generate enough tokens - use "O" for remaining
            predictions.append("O")
    
    return predictions, retrieved_examples, prompt

test_example = dataset['test'][5]
predictions, retrieved, prompt_used = deer_ner_inference(
    test_example, 
    retriever, 
    model, 
    tokenizer,
    dataset['train'][:3000],
    k=3
)

print("Input:")
print(" ".join(test_example['tokens'][:15]))

print("DEER Predictions (from Phi-3):")
print(" ".join([f"{tok}|{pred}" for tok, pred in zip(test_example['tokens'][:15], predictions[:15])]))

print("Ground Truth:")
true_tags = [label_names[tag] for tag in test_example['ner_tags'][:15]]
print(" ".join([f"{tok}|{tag}" for tok, tag in zip(test_example['tokens'][:15], true_tags)]))

print("Retrieved Examples (via Label-Guided Retrieval):")
for i, example in enumerate(retrieved, 1):
    entity_types = set()
    for tag_id in example['ner_tags']:
        tag = label_names[tag_id]
        if tag != 'O':
            entity_types.add(tag.split('-')[1])
    print(f"  Example {i}: contains {', '.join(entity_types) if entity_types else 'no entities'}")

In [ ]:
def evaluate_deer_ner(test_dataset, retriever, model, tokenizer, train_dataset, n_samples=50):
    """
    Оценка NER performance с DEER + Phi-3
    """
    all_predictions = []
    all_ground_truth = []
    
    for i in range(min(n_samples, len(test_dataset))):
        test_example = test_dataset[i]
        
        # DEER inference
        predictions, _, _ = deer_ner_inference(
            test_example, retriever, model, tokenizer, train_dataset, k=3
        )
        
        # Ground truth
        true_tags = [label_names[tag] for tag in test_example['ner_tags']]
        
        all_predictions.append(predictions)
        all_ground_truth.append(true_tags)
    
    # Вычисляем метрики
    try:
        f1 = f1_score(all_ground_truth, all_predictions)
        precision = precision_score(all_ground_truth, all_predictions)
        recall = recall_score(all_ground_truth, all_predictions)
    except:
        # Fallback если seqeval не работает
        f1, precision, recall = 0.0, 0.0, 0.0
    
    return {
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'n_samples': n_samples
    }

ner_metrics = evaluate_deer_ner(
    dataset['test'],
    retriever,
    model,
    tokenizer,
    dataset['train'][:3000],
    n_samples=50
)

print("="*60)
print("NER METRICS: Phi-3 + DEER on WNUT-17")
print("="*60)
print(f"\nF1-Score:  {ner_metrics['f1']:.3f}")
print(f"Precision: {ner_metrics['precision']:.3f}")
print(f"Recall:    {ner_metrics['recall']:.3f}")
print(f"Samples:   {ner_metrics['n_samples']}")

In [ ]:
def evaluate_retrieval_quality(test_dataset, train_dataset, retriever, k=3, n_samples=100):
    deer_entity_coverage = []  # Percentage of covered entity types (DEER)
    random_entity_coverage = []  # Baseline: random retrieval
    
    deer_token_relevance = []  # Average token relevance (DEER)
    random_token_relevance = []  # Baseline
    
    for i in range(min(n_samples, len(test_dataset))):
        test_example = test_dataset[i]
        test_tokens = test_example['tokens']
        test_tags = [label_names[tag] for tag in test_example['ner_tags']]
        
        # Determine entity types in the test example
        test_entity_types = set()
        for tag in test_tags:
            if tag != 'O':
                test_entity_types.add(tag.split('-')[1])
        
        if not test_entity_types:  # Skip if no entities
            continue
        
        # DEER retrieval
        deer_examples = retriever.retrieve_examples(test_tokens, train_dataset, k=k)
        
        # Random retrieval (baseline)
        random_indices = np.random.choice(len(train_dataset), size=k, replace=False)
        random_examples = [train_dataset[idx] for idx in random_indices]
        
        # Function to compute coverage
        def compute_coverage(examples):
            covered_types = set()
            total_relevance = 0.0
            
            for example in examples:
                tags = [label_names[tag] for tag in example['ner_tags']]
                for tag in tags:
                    if tag != 'O':
                        entity_type = tag.split('-')[1]
                        if entity_type in test_entity_types:
                            covered_types.add(entity_type)
                
                # Compute average token relevance
                for token in example['tokens']:
                    max_relevance = max([deer_stats.get_token_score(token, et) 
                                        for et in test_entity_types] + [0.0])
                    total_relevance += max_relevance
            
            coverage = len(covered_types) / len(test_entity_types) if test_entity_types else 0.0
            avg_relevance = total_relevance / (k * 20) if k > 0 else 0.0  # ~20 tokens per example
            
            return coverage, avg_relevance
        
        deer_cov, deer_rel = compute_coverage(deer_examples)
        random_cov, random_rel = compute_coverage(random_examples)
        
        deer_entity_coverage.append(deer_cov)
        random_entity_coverage.append(random_cov)
        deer_token_relevance.append(deer_rel)
        random_token_relevance.append(random_rel)
    
    results = {
        'deer_coverage': np.mean(deer_entity_coverage),
        'random_coverage': np.mean(random_entity_coverage),
        'deer_relevance': np.mean(deer_token_relevance),
        'random_relevance': np.mean(random_token_relevance),
    }
    
    return results

print("Evaluating retrieval quality (this will take some time)...")
eval_results = evaluate_retrieval_quality(
    dataset['validation'], 
    dataset['train'][:5000],  # Using a subset for speed
    retriever, 
    k=3, 
    n_samples=100
)

print("\n" + "="*60)
print("RETRIEVAL EVALUATION RESULTS")
print("="*60)
print(f"\nEntity Type Coverage (how well the selected examples cover entity types):")
print(f"  DEER:   {eval_results['deer_coverage']:.1%}")
print(f"  Random: {eval_results['random_coverage']:.1%}")
print(f"  Improvement: +{(eval_results['deer_coverage'] - eval_results['random_coverage']) * 100:.1f}%")

print(f"\nToken Relevance (average token relevance):")
print(f"  DEER:   {eval_results['deer_relevance']:.4f}")
print(f"  Random: {eval_results['random_relevance']:.4f}")
print(f"  Improvement: +{((eval_results['deer_relevance'] / eval_results['random_relevance']) - 1) * 100:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Comparison bar chart - Coverage
ax1 = axes[0, 0]
methods = ['DEER\n(Label-Guided)', 'Random\nBaseline']
coverage_values = [eval_results['deer_coverage'], eval_results['random_coverage']]
colors = ['#2ecc71', '#e74c3c']
bars = ax1.bar(methods, coverage_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('Entity Type Coverage', fontsize=12, fontweight='bold')
ax1.set_title('(A) Entity Type Coverage Comparison', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 1.0)
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

for bar, value in zip(bars, coverage_values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{value:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 2. Token Relevance comparison
ax2 = axes[0, 1]
relevance_values = [eval_results['deer_relevance'], eval_results['random_relevance']]
bars = ax2.bar(methods, relevance_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Average Token Relevance Score', fontsize=12, fontweight='bold')
ax2.set_title('(B) Token Relevance Comparison', fontsize=13, fontweight='bold')

for bar, value in zip(bars, relevance_values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.001,
             f'{value:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 3. Token-level statistics (top informative tokens per entity type)
ax3 = axes[1, 0]

# Collect top-5 tokens for each entity type
entity_types = ['PER', 'ORG', 'LOC', 'MISC']
top_tokens_data = []

for entity_type in entity_types:
    tokens_scores = []
    for token in deer_stats.token_counts.keys():
        score = deer_stats.get_token_score(token, entity_type)
        if score > 0:
            tokens_scores.append((token, score))
    
    # Top-3 tokens
    tokens_scores.sort(key=lambda x: x[1], reverse=True)
    top_3 = tokens_scores[:3]
    
    for token, score in top_3:
        top_tokens_data.append({
            'entity': entity_type,
            'token': token,
            'score': score
        })

# Create heatmap
if top_tokens_data:
    import pandas as pd
    df_tokens = pd.DataFrame(top_tokens_data)
    
    # Pivot for heatmap
    pivot_data = []
    for entity_type in entity_types:
        entity_data = df_tokens[df_tokens['entity'] == entity_type]
        if len(entity_data) > 0:
            pivot_data.append(entity_data.head(3)['score'].tolist() + [0] * (3 - len(entity_data)))
    
    if pivot_data:
        pivot_array = np.array(pivot_data)
        sns.heatmap(pivot_array, annot=True, fmt='.3f', cmap='YlOrRd', 
                   xticklabels=['Token 1', 'Token 2', 'Token 3'],
                   yticklabels=entity_types, ax=ax3, cbar_kws={'label': 'P(token|entity)'})
        ax3.set_title('Most Informative Tokens per Entity Type', fontsize=13, fontweight='bold')
        ax3.set_xlabel('Top Tokens', fontsize=11)
        ax3.set_ylabel('Entity Type', fontsize=11)

# 4. Improvement visualization
ax4 = axes[1, 1]

metrics = ['Coverage', 'Relevance']
improvements = [
    (eval_results['deer_coverage'] - eval_results['random_coverage']) * 100,
    ((eval_results['deer_relevance'] / eval_results['random_relevance']) - 1) * 100
]

bars = ax4.barh(metrics, improvements, color='#3498db', alpha=0.7, edgecolor='black', linewidth=2)
ax4.set_xlabel('Improvement over Random Baseline (%)', fontsize=12, fontweight='bold')
ax4.set_title('DEER Improvements', fontsize=13, fontweight='bold')
ax4.axvline(x=0, color='black', linestyle='-', linewidth=1)

for bar, value in zip(bars, improvements):
    width = bar.get_width()
    ax4.text(width + 1, bar.get_y() + bar.get_height()/2.,
             f'+{value:.1f}%', ha='left', va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('deer_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
test_idx = 42
test_example = dataset['validation'][test_idx]
test_tokens = test_example['tokens']
test_tags = [label_names[tag] for tag in test_example['ner_tags']]

print("Test Example:")
print(" ".join([f"{token}|{tag}" for token, tag in zip(test_tokens, test_tags)]))

# Determining entity types in the test example
entity_types_in_test = set()
for tag in test_tags:
    if tag != 'O':
        entity_types_in_test.add(tag.split('-')[1])

print(f"Entity types in test: {', '.join(entity_types_in_test)}")

# Showing the most informative tokens for these entity types
print("Most informative tokens for these entity types:")
for entity_type in entity_types_in_test:
    informative_tokens = []
    for token in test_tokens:
        score = deer_stats.get_token_score(token, entity_type)
        if score > 0.01:
            informative_tokens.append((token, score))
    
    informative_tokens.sort(key=lambda x: x[1], reverse=True)
    print(f"   {entity_type}: {', '.join([f'{t}({s:.3f})' for t, s in informative_tokens[:3]])}")

# DEER retrieval
deer_examples = retriever.retrieve_examples(test_tokens, dataset['train'][:5000], k=3)

print("DEER Retrieved Examples:")
for i, example in enumerate(deer_examples, 1):
    tokens = example['tokens']
    tags = [label_names[tag] for tag in example['ner_tags']]
    
    # Highlight entities
    formatted = []
    for token, tag in zip(tokens, tags):
        if tag != 'O':
            formatted.append(f"**{token}**[{tag}]")
        else:
            formatted.append(token)
    
    print(f"\n   Example {i}:")
    print(f"   {' '.join(formatted[:20])}...")  # First 20 tokens

# Random retrieval for comparison
random_indices = np.random.choice(len(dataset['train']), size=3, replace=False)
random_examples = [dataset['train'][idx] for idx in random_indices]

print("Random Baseline Examples:")
for i, example in enumerate(random_examples, 1):
    tokens = example['tokens']
    tags = [label_names[tag] for tag in example['ner_tags']]
    
    formatted = []
    for token, tag in zip(tokens, tags):
        if tag != 'O':
            formatted.append(f"**{token}**[{tag}]")
        else:
            formatted.append(token)
    
    print(f"\n   Example {i}:")
    print(f"   {' '.join(formatted[:20])}...")